# Credit card fraud detection

## CC_Fraud_DATA_Prep as Data Preparation Script


Since the original ipynb script got more messy and harder to keep clean, this script aims to preprocess the dataset's (train & test) and to exclude all the feature engineering steps from the main script.

## Introduction

## Introduction

The machine learning project aims to detect fraudulent credit card transaction, which is a growing problem for the banking industry and also for consumers. Due to growing scam and social engineering activities more people are victims. Therefore there are multiple stakeholders (regulatory authorities EBA & ECB, banks, consumers) interested in a sufficient fraud prevention and detection. 



The credit card data (most likely synthetic data) is downloaded from Hugging face, without providing any descriptions.
Nevertheless the used variables are quit obvious, by their structure and their content. 
So the qualitative data description will be conducted on industrial experience and best guesses.


In [6]:
# Importing standard packages
import polars as pl # Faster then pandas, 
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt # visualization
#import sklearn as 

from geopy.distance import distance #, geodesic, great_circle # Feature Engineering module for Geodata

In [7]:
# reading in the datasets
# Read CSV files with Polars
#cc_train = pl.read_csv("/home/sarima/Desktop/UNI Olsztyn /5# Machine Learning/ML_Project/credit_card_transaction_train.csv")
#cc_test  = pl.read_csv("/home/sarima/Desktop/UNI Olsztyn /5# Machine Learning/ML_Project/credit_card_transaction_test.csv")


Threshold in noon (2020, 6, 21)

In [ ]:
# reading in the datasets
# Read CSV files with Polars
cc = pl.read_csv("/home/sarima/Desktop/UNI Olsztyn /5# Machine Learning/ML_Project/credit_card_transaction_test.csv")
#cc_test  = pl.read_csv("/home/sarima/Desktop/UNI Olsztyn /5# Machine Learning/ML_Project/credit_card_transaction_test.csv")

# calculation for training, exporting
# then, copy of script, reading cc as cc_test. 
# seperate script/data import/ output fastest way (because of Outlier calc etc, not together and then split as consideration!) 

Since amount is an issue in the results, I got in the first run an almost perfect predictability (the problem of bad synthetic data) I will randomly change (modify it with noise) a part (70 percentage) of the Amount values and also for the merchant lat and long.


In [ ]:
np.random.seed(42)
random_multipliers = np.random.uniform(0.5, 9.0, size=len(cc))

cc = cc.with_columns(
    pl.when(np.random.random(len(cc)) < 0.7)
    .then(pl.col('amt') * random_multipliers)
    .otherwise(pl.col('amt'))
    .alias('amt')
)
"""
merchants = cc['merchant'].unique().to_list()

noise_df = pl.DataFrame({
    'merchant': merchants,
    'lat_noise': np.random.uniform(-5, 5, size=len(merchants)).astype(int),
    'long_noise': np.random.uniform(-5, 5, size=len(merchants)).astype(int)
})

# Join to add jitter (instead of map_elements)
cc = cc.join(noise_df, on='merchant', how='left')

# Apply jitter
cc = cc.with_columns(
    (pl.col('merch_lat') + pl.col('lat_noise')).alias('merch_lat'),
    (pl.col('merch_long') + pl.col('long_noise')).alias('merch_long')
)
del noise_df
# Drop jitter columns
cc = cc.drop(['lat_noise', 'long_noise'])
"""


ShapeError: could not create a new DataFrame: height of column 'lat_noise' (693) does not match height of column 'merchant' (50)

Additionally there is taken fraud data from the FBI fillings published via the pdf documentation.
State  data from the FBI.pdf, p.29
Losses per 100,000 citizens in each state

In [10]:
crime_data = pl.DataFrame({
    'State': ['District of Columbia', 'Nevada', 'California', 'New Jersey', 'Arizona', 
              'Alaska', 'Montana', 'South Dakota', 'Utah', 'Florida', 'New York', 
              'Washington', 'Hawaii', 'Maryland', 'Delaware', 'Minnesota', 'Massachusetts', 
              'Texas', 'Connecticut', 'Oregon', 'Kansas', 'Colorado', 'Virginia', 
              'Rhode Island', 'Pennsylvania', 'Georgia', 'Illinois', 'Idaho', 'Indiana', 
              'Wyoming', 'Tennessee', 'South Carolina', 'North Carolina', 'New Mexico', 
              'Nebraska', 'Michigan', 'Missouri', 'New Hampshire', 'Alabama', 'Iowa', 
              'North Dakota', 'Louisiana', 'Ohio', 'Oklahoma', 'Wisconsin', 'Arkansas', 
              'Puerto Rico', 'Vermont', 'Maine', 'West Virginia', 'Mississippi', 'Kentucky'],

    'State_ID': [ 'DC',    'NV',     'CA',     'NJ',     'AZ',     'AK',     'MT',     'SD',     'UT',     'FL',     'NY',
     'WA',     'HI',     'MD',     'DE',     'MN',     'MA',     'TX',     'CT',     'OR',     'KS',     'CO',     'VA',
     'RI',    'PA',     'GA',     'IL',     'ID',     'IN',     'WY',     'TN',     'SC',     'NC',     'NM',     'NE',
     'MI',     'MO',     'NH',    'AL',     'IA',     'ND',     'LA',     'OH', 'OK',     'WI',     'AR',     'PR',
     'VT',     'ME',     'WV',     'MS',     'KY'],
     
    'Loss in $/Capita': [6795914, 6292550, 5542009, 4748238, 4364657, 4332018, 4021353, 3900228,
             3869729, 3868631, 3831931, 3695066, 3603978, 3584328, 3428347, 3380137,
             3369186, 3348973, 3338719, 3213809, 3202070, 3192143, 3041335, 2882110,
             2779999, 2729130, 2675478, 2577030, 2364534, 2353556, 2261914, 2232240,
             2168543, 2134317, 2051237, 2026907, 1991645, 1938461, 1888622, 1865588,
             1726240, 1711639, 1674584, 1651948, 1557861, 1518551, 1479384, 1361957,
             1359051, 1211587, 1093451, 1076986]
})

Even though this specific dataset seems to be synthetically derived and the names and personal data is most likelly not critical to process, in the next steps there will be an anomisation of personel data for the following reasons:

- Depending on the contract between the bank and a third party consulting firm, the data might be depriaciated by the personal information. Therefore there is no simple deletion of the First and Last name of the data, but by an hashing algorithm an anomisation but still conserving client data informations of multiple credit cards. In reality it seems to be a realistic scenario that a victim loosing his credit card information, by social engineering or different scamming attacs might also loose multiple credit card information and therefore there might be a correlation with a causal background.

- The dataset contains also data for gender and date of birth, which is from an ethical viewpoint critical by a following discrimination. The results of the algorithm could suggest that a specific gender or age is more vulnerable to scam and therefore could be excluded from banking business or get an risk premium, by simply being in that socio economic group. In real application this should be considered.  

In [11]:
# hashing name data for anominization and deleting prior columns
cc = cc.with_columns(
    (pl.col("first").cast(pl.Utf8) + "|" + pl.col("last").cast(pl.Utf8)).alias("Owner_ID")
)

cc = cc.with_columns(
    pl.col("Owner_ID").hash().cast(pl.UInt64).alias("Owner_ID")
)

cc = cc.drop(['first', 'last']) 

## First Overview of the Data

### Qualitative Description of the DATA, What the Variables mean.



## Feature Engineering

Since some feature should be considered together, as input into the model are the engineered features instead of the original variables used.

Considering longitude and latitude in combination (both vor the victim, as well as for the merchant) results in information about the distance, whereas only each variable on their own is not sufficent.

Furthermore it is also reducing the dimensionality, by using less input variables within the model.

#Data Problem
The amt is unfortunately problematic, since there is a clear pattern for the relation. To make it more realistic randomly amt will be changed....

In [12]:
# Check unique amounts in fraud
print("\nUnique amt fraud:")
print(cc.filter(pl.col('is_fraud') == 1)['amt'].value_counts().head(10))

# Check unique amounts in non-fraud
print("\nUnique amt in non-fraud:")
print(cc.filter(pl.col('is_fraud') == 0)['amt'].value_counts().head(10))


Unique amt fraud:
shape: (10, 2)
┌────────┬───────┐
│ amt    ┆ count │
│ ---    ┆ ---   │
│ f64    ┆ u32   │
╞════════╪═══════╡
│ 316.64 ┆ 1     │
│ 299.89 ┆ 1     │
│ 20.08  ┆ 1     │
│ 326.64 ┆ 1     │
│ 867.53 ┆ 1     │
│ 739.29 ┆ 1     │
│ 218.02 ┆ 1     │
│ 751.26 ┆ 1     │
│ 309.99 ┆ 1     │
│ 776.37 ┆ 1     │
└────────┴───────┘

Unique amt in non-fraud:
shape: (10, 2)
┌────────┬───────┐
│ amt    ┆ count │
│ ---    ┆ ---   │
│ f64    ┆ u32   │
╞════════╪═══════╡
│ 26.17  ┆ 72    │
│ 169.32 ┆ 8     │
│ 79.2   ┆ 85    │
│ 211.91 ┆ 7     │
│ 197.33 ┆ 6     │
│ 119.53 ┆ 20    │
│ 324.29 ┆ 1     │
│ 306.04 ┆ 2     │
│ 68.25  ┆ 71    │
│ 190.74 ┆ 10    │
└────────┴───────┘


In [13]:
# Therefore we add noise, to make it more realistic:
import numpy as np
np.random.seed(42)
cc= cc.with_columns(
    (pl.col('amt') * np.random.uniform(0.5, 9.0, size=len(cc))).alias('amt')
)

I.1) Engineered Feature 
Constructing a Risk Rating of each state by using FBI Data on Fraud occurence. If there are states in which due to governmental reasons, like regulation or technical infrasturcture. This feature might be an exogenous variable on our fraud flag. 

In [14]:
# combining States (governmental reasons) and their fraud losses/per Capita (extra dataset) as Risk-Rating Variable
# State_Risk_Rating

crime = crime_data.select([
    pl.col("State_ID"),
    pl.col("Loss in $/Capita").alias("State_Risk_Rating")
])

cc = cc.join(
    crime,
    left_on="state",
    right_on="State_ID",
    how="left"
)
del crime_data

I.2) Engineered Feature
Using the geolocation data, to check if the adress of the client/victim and the merchant is distant. It might indicate either travelling or online shopping, but also suspicous behaviour. An card usage abroad in south east asia, which did not happen before could be at least suspicous, contacting a client and asking for current travel could be a safety feature.


## Problem!! dist_client_merchant was most likely used to generate is_fraud !! 
Therefore we cannot use it!

In [15]:
# Using client and merchant distance, rowwise
# from geopy.distance import distance 

cc = cc.with_columns(
    pl.struct(['lat', 'long', 'merch_lat', 'merch_long'])
    .map_elements(
        lambda row: distance((row['lat'], row['long']), 
                            (row['merch_lat'], row['merch_long'])).km,
        return_dtype=pl.Float64
    )
    .alias('dist_client_merchant')
)

I.3) Engineered Feature
This feature is measering the distance of the previous card transaction to the current card transaction in relation to time. Even though due to travel or online shopping, this information does not tell for sure any fraudulent behaviour, but it shows some unusual behaviour, which is at least an indicator. In comparison to the second feature it is a more complex, but also more causal.

In [16]:
# transforming str dtype int dateformat

cc = cc.with_columns([
    pl.col('trans_date_trans_time').str.to_datetime().alias('trans_date_trans_time'), # format change
    ])

cc = cc.with_columns([
    pl.col('trans_date_trans_time').dt.date().alias('trans_date'), 
    pl.col('trans_date_trans_time').dt.week().alias('trans_week') # extract transaction date, week
    ])


I.4) Transaction time difference trans_time_diff

In [17]:
# Using merchant distance and time between transactions for each card
cc = cc.sort(['cc_num', 'trans_date_trans_time'])

# Create lagged values for each card
cc = cc.with_columns([
    pl.col('merch_lat').shift(1).over('cc_num').alias('prev_merch_lat'),
    pl.col('merch_long').shift(1).over('cc_num').alias('prev_merch_long'),
    pl.col('trans_date_trans_time').shift(1).over('cc_num').alias('prev_trans_time')
])

# distance in km, per transaction
cc = cc.with_columns(
    pl.struct(['merch_lat', 'merch_long', 'prev_merch_lat', 'prev_merch_long'])
    .map_elements(
        lambda row: distance(
            (row['merch_lat'], row['merch_long']),
            (row['prev_merch_lat'], row['prev_merch_long'])
        ).km if row['prev_merch_lat'] is not None else 0.0, # None problem with lagged data
    ).alias('dist_prev_current_trans') # polars got a  problem with integer and Float64, therefore 0.0!
)


# transaction time difference
# maybe engineered feature on its own?
cc = cc.with_columns(
    pl.struct(['trans_date_trans_time', 'prev_trans_time'])
    .map_elements(
        lambda row: (row['trans_date_trans_time'] - row['prev_trans_time']).total_seconds()
        if row['prev_trans_time'] is not None else 0.0, # None problem with lagged data
    ).alias('trans_time_diff')
)
cc = cc.drop(['prev_merch_lat', 'prev_merch_long','prev_trans_time'])

I.5) Travel Distance in km per second

In [18]:

cc = cc.with_columns(
    (pl.col('dist_prev_current_trans') / pl.col('trans_time_diff'))
    .fill_nan(0.0)           #  0/0 case
    .replace(float('inf'), 0.0)  # division by trans_time_diff == 0
  #  .fill_null(0.0)          # Handle any remaining nulls
    .alias('travel_time_km')
)

cc=cc.drop(['dist_prev_current_trans'])

# Unit Kilometers per second (km/s) (because more precise), per hour is unrealistic if 

I.6) Daily Transactions / Weekly Transactions per CC to check for frequent card usage.

If a card is heavily used this might impact fraud. A lot of transaction makes it harder to keep track of fraudulent transaction: 
(1 Cent transfer, to check if cc data is valid, note: the min Transaction is a dollar, therefore no such feature)
Calculation will be done daily / weekly. Not for the full period! Daily is from the perspective reasonable, because fraudulent transaction will most likely occur in a short time span if scammers want to maximize their cc output. 
weekly is on the other hand reasonable, because it is often (anglosaxian countries) that billing of utitilities/salary payment and   also credit card invoices are sent on a weekly bases. Therefore checks by the victims will most likely also occur in this time span.

In [19]:
cc = cc.with_columns(
    pl.len().over(['cc_num', 'trans_date']).alias('daily_trans')
).with_columns(
    pl.col('daily_trans').shift(1).over(['cc_num', 'trans_date']).fill_null(0).alias('daily_trans_lagged') # Missings with 0 instead Null for error handling
)

"""
problem of future information in current row!
cc = cc.join(
    cc.group_by(['cc_num', 'trans_week']).agg(
        pl.count().alias('weekly_transactions')
    ),
    on=['cc_num', 'trans_week'],
    how='left'
)
"""

"\nproblem of future information in current row!\ncc = cc.join(\n    cc.group_by(['cc_num', 'trans_week']).agg(\n        pl.count().alias('weekly_transactions')\n    ),\n    on=['cc_num', 'trans_week'],\n    how='left'\n)\n"

I.7) Online Shopping Binary Variable

In [20]:
# Therefore everything with *net is considered as online shopping, everything else physiscal bought via terminal
# (very easy assumption) unlikely in reality.
# But maybe Mastercard/Visa/Amex provide in reality data if card is used by a terminal / or via websecure..
# ['shopping_net','misc_net','grocery_net']
# Binary Variable for online shopping marker
cc = cc.with_columns(
    pl.when(pl.col('category').is_in(['shopping_net','misc_net','grocery_net']))
    .then(0)
    .otherwise(1)
    .alias('net_binary')
)

## Time related data checks...

If there is no strictly cross sectional data (which is not, in most cases), the question arises during the time span any structural changes are happening and if we are considering them appropriate. Since the training data spans the period, over 18 months (2019, 1, 1 until 2020, 6, 21) and the test data another 6 months (2020, 6, 21 until 2020, 12, 31) it might be a rather short period, but nevertheless a period in which enough could happen with explainable effects. 
Examples could be structural breaks due to new regulatory or technical advancements, which we don't now expicitly by observing the data for each US State. Another argument could be the start or end of a group or person with highly "motivated" scam activities.  

In [21]:
# feature considering rural and metropolitan area (city population), but also the city name.
# frequent crime cities are important, but the population can be densed in the information.

# Village < 5.000 < 'Small_Town' < 50.000 < 'Town' < 500.000 < 'City' < 999.999 < 'Metropolitan_Area'  
# Numbers are debatable and choosen by personal estimation! (-> Here is a big attacking point)

cc = cc.with_columns(
    pl.when(pl.col('city_pop') < 5000)
    .then(pl.lit('Village'))
    .when(pl.col('city_pop') < 50000)
    .then(pl.lit('Small_Town'))
    .when(pl.col('city_pop') < 500000)
    .then(pl.lit('Town'))
    .when(pl.col('city_pop') < 1000000)
    .then(pl.lit('City'))
    .otherwise(pl.lit('Metropolitan_Area'))
    .alias('Population_Density')
)
# without pl.lit is an error... Check if time is left, about my package versions

# Histograms are usefull in considering the buckets graphical analysis of Raw Data. Is in a different part/other script. 
# To maintain some clean order and structure, because this script gets confusing...
"""
plt.hist(cc['city_pop'], bins=50, alpha=0.7, color='blue', edgecolor='black')
plt.xlabel('city_pop')
plt.ylabel('Frequency')
plt.title('City Population Distribution')
plt.show()
"""


"\nplt.hist(cc['city_pop'], bins=50, alpha=0.7, color='blue', edgecolor='black')\nplt.xlabel('city_pop')\nplt.ylabel('Frequency')\nplt.title('City Population Distribution')\nplt.show()\n"

I.8) Birth Year Categorization

In [22]:
# changing type from string
cc = cc.with_columns(
    pl.col('dob').str.strptime(pl.Date, format="%Y-%m-%d")  # Adjust format as needed
)


In [23]:
# In similiar manner the variable Date of Birth is using a generational categorization
# Create generation variable
cc = cc.with_columns(
    pl.col('dob').dt.year().alias('birth_year')
).with_columns(
    pl.when(pl.col('birth_year') < 1946)
    .then(pl.lit('War_generation'))
    .when(pl.col('birth_year') < 1965)
    .then(pl.lit('Boomer'))
    .when(pl.col('birth_year') < 1981)
    .then(pl.lit('Gen X'))
    .when(pl.col('birth_year') < 1997)
    .then(pl.lit('Millennials'))
    .when(pl.col('birth_year') < 2013)
    .then(pl.lit('Gen Z'))
    .alias('generation')
).drop(['dob'])

# checking Birth year vs categorical in modelling

I.9) Amount per client in relation to IQR
-> problematic, because usage of global variables!!!
-> deleted

Combined Features 

In [24]:
# Combined Features 
# Time and Travel time for card usage relevant if no online shopping is happening!
cc = cc.with_columns(
    (pl.col('net_binary') * pl.col('travel_time_km')).alias('net_binary_comb_travel_time')
)
# Same dependency of online and transaction time
cc = cc.with_columns(
    (pl.col('net_binary') * pl.col('trans_time_diff')).alias('net_binary_comb_trans_time')
)

"""
# Amount of spending relevant if it is an Outlier
cc = cc.with_columns(
    (pl.col('Amount_Outlier_bin') * pl.col('amt')).alias('Amount_Outlier_bin_comb_amt')
)
"""


"\n# Amount of spending relevant if it is an Outlier\ncc = cc.with_columns(\n    (pl.col('Amount_Outlier_bin') * pl.col('amt')).alias('Amount_Outlier_bin_comb_amt')\n)\n"

In [25]:
# 
cc=cc.drop('street','city','state','zip','lat','long','merch_zipcode',
'unix_time', 'Unnamed: 0','merch_lat', 'merch_long','trans_week')

In [26]:
# Reordering ID columns to the front columns
cc = cc.select([
    pl.col('Owner_ID'),
    pl.col('cc_num'),
    pl.col('trans_num'),
    pl.col('trans_date_trans_time'),    
    pl.all().exclude(['Owner_ID', 'cc_num', 'trans_num','trans_date_trans_time'])
])



In [ ]:
## exporting as parquet for faster usage in future
#cc.write_parquet("credit_card_transactions.parquet")
# important test input and export change here!
cc.head(10)

# If Train Data is written
cc.write_parquet("credit_card_trans_test.parquet")



In [28]:
# If Test Data is written
# cc.write_parquet("credit_card_trans_test.parquet")

In [29]:
# Column names
print(f"Columns: {cc.columns}")

Columns: ['Owner_ID', 'cc_num', 'trans_num', 'trans_date_trans_time', 'merchant', 'category', 'amt', 'gender', 'city_pop', 'job', 'is_fraud', 'State_Risk_Rating', 'dist_client_merchant', 'trans_date', 'trans_time_diff', 'travel_time_km', 'daily_trans', 'daily_trans_lagged', 'net_binary', 'Population_Density', 'birth_year', 'generation', 'net_binary_comb_travel_time', 'net_binary_comb_trans_time']
